<a href="https://colab.research.google.com/github/iescobedo5/Computer-Vision/blob/main/YOLO30K_DEEP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q ultralytics deep-sort-realtime opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 84.9 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

import cv2
import time
from pathlib import Path

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/new_UA/yolo_results30K/uadetrac_subset30k_e25/weights/best.pt"

IMAGE_DIR = Path("/content/drive/MyDrive/MVI_39031")
SAVE_DIR = Path("/content/drive/MyDrive/UA/yolo30k_deepsort/MVI_39031")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TRACK_TXT = SAVE_DIR / "yolo30k_deepsort_pred.txt"
VIDEO_OUT = SAVE_DIR / "yolo30k_deepsort_output.mp4"

CONF_THRESH = 0.25

model = YOLO(MODEL_PATH)

In [ ]:
tracker = DeepSort(
    max_age=30,
    n_init=3,
    max_cosine_distance=0.2,
    nn_budget=100
)

In [ ]:
image_files = sorted(
    list(IMAGE_DIR.glob("*.jpg")) +
    list(IMAGE_DIR.glob("*.png"))
)

print("Number of frames:", len(image_files))

if len(image_files) == 0:
    raise ValueError(f"No images found in {IMAGE_DIR}")

first_img = cv2.imread(str(image_files[0]))
h, w = first_img.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_writer = cv2.VideoWriter(str(VIDEO_OUT), fourcc, 25, (w, h))

Number of frames: 1470


In [ ]:
start_time = time.time()

with open(TRACK_TXT, "w") as f:

    for frame_idx, img_path in enumerate(image_files, start=1):

        frame = cv2.imread(str(img_path))

        results = model(frame, conf=CONF_THRESH, verbose=False)[0]

        detections = []

        if results.boxes is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            scores = results.boxes.conf.cpu().numpy()
            classes = results.boxes.cls.cpu().numpy()

            for box, score, cls in zip(boxes, scores, classes):
                x1, y1, x2, y2 = box

                width = x2 - x1
                height = y2 - y1

                # DeepSORT format:
                # ([left, top, width, height], confidence, class)
                detections.append(
                    ([x1, y1, width, height], float(score), int(cls))
                )

        tracks = tracker.update_tracks(detections, frame=frame)

        for track in tracks:
            if not track.is_confirmed():
                continue

            track_id = track.track_id
            l, t, r, b = track.to_ltrb()

            x = l
            y = t
            width = r - l
            height = b - t

            # MOT format:
            # frame, id, x, y, w, h, conf, x, y, z
            f.write(
                f"{frame_idx},{track_id},{x:.2f},{y:.2f},"
                f"{width:.2f},{height:.2f},1,-1,-1,-1\n"
            )

            # Draw box
            cv2.rectangle(
                frame,
                (int(l), int(t)),
                (int(r), int(b)),
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"ID {track_id}",
                (int(l), int(t) - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

        video_writer.write(frame)

        if frame_idx % 100 == 0:
            print(f"Processed {frame_idx}/{len(image_files)} frames")

video_writer.release()

total_time = time.time() - start_time
fps = len(image_files) / total_time

print("Done!")
print(f"Saved tracking txt to: {TRACK_TXT}")
print(f"Saved video to: {VIDEO_OUT}")
print(f"Total time: {total_time:.2f} seconds")
print(f"FPS: {fps:.2f}")

Processed 100/1470 frames
Processed 200/1470 frames
Processed 300/1470 frames
Processed 400/1470 frames
Processed 500/1470 frames
Processed 600/1470 frames
Processed 700/1470 frames
Processed 800/1470 frames
Processed 900/1470 frames
Processed 1000/1470 frames
Processed 1100/1470 frames
Processed 1200/1470 frames
Processed 1300/1470 frames
Processed 1400/1470 frames
Done!
Saved tracking txt to: /content/drive/MyDrive/UA/yolo30k_deepsort/MVI_39031/yolo30k_deepsort_pred.txt
Saved video to: /content/drive/MyDrive/UA/yolo30k_deepsort/MVI_39031/yolo30k_deepsort_output.mp4
Total time: 151.48 seconds
FPS: 9.70


In [ ]:
!rm -rf /content/TrackEval
!git clone https://github.com/JonathonLuiten/TrackEval.git
%cd /content/TrackEval
!pip install scipy matplotlib tqdm

Cloning into 'TrackEval'...
remote: Enumerating objects: 924, done.
remote: Counting objects: 100% (375/375), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 924 (delta 340), reused 315 (delta 315), pack-reused 549 (from 1)
Receiving objects: 100% (924/924), 393.80 KiB | 15.15 MiB/s, done.
Resolving deltas: 100% (619/619), done.
/content/TrackEval


In [ ]:
!mkdir -p /content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt
!cp /content/drive/MyDrive/UA_DETRAC_GT/gt.txt /content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt/gt.txt

In [ ]:
# add prediction file to TrackEval
!mkdir -p /content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/yolo30k_deepsort/data
!cp /content/drive/MyDrive/UA/yolo30k_deepsort/MVI_39031/yolo30k_deepsort_pred.txt /content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/yolo30k_deepsort/data/MVI_39031.txt

In [ ]:
from pathlib import Path

seqmap_dir = Path("/content/TrackEval/data/gt/mot_challenge/seqmaps")
seqmap_dir.mkdir(parents=True, exist_ok=True)

seqmap_file = seqmap_dir / "UA-DETRAC-train.txt"

seqmap_file.write_text("name\nMVI_39031\n")

print(seqmap_file.read_text())

name
MVI_39031



In [ ]:
import numpy as np
from pathlib import Path

gt_path = Path("/content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt/gt.txt")
pred_path = Path("/content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/yolo30k_deepsort/data/MVI_39031.txt")

gt = np.loadtxt(gt_path, delimiter=",")
pred = np.loadtxt(pred_path, delimiter=",")

gt_max_frame = int(gt[:, 0].max())

# Handle case where pred is empty
if pred.size > 0:
    pred_max_frame = int(pred[:, 0].max())
else:
    pred_max_frame = 0 # If no predictions, max frame is 0

print("GT max frame:", gt_max_frame)
print("Prediction max frame:", pred_max_frame)

seq_length = max(gt_max_frame, pred_max_frame)

seqinfo_path = Path("/content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/seqinfo.ini")

seqinfo_text = f"""
[Sequence]
name=MVI_39031
imDir=img1
frameRate=25
seqLength={seq_length}
imWidth=960
imHeight=540
imExt=.jpg
"""

seqinfo_path.write_text(seqinfo_text.strip())

print(seqinfo_path.read_text())

GT max frame: 1470
Prediction max frame: 1470
[Sequence]
name=MVI_39031
imDir=img1
frameRate=25
seqLength=1470
imWidth=960
imHeight=540
imExt=.jpg


In [ ]:
%cd /content/TrackEval

!grep -R "np.float" -n trackeval
!find trackeval -type f -name "*.py" -exec sed -i 's/np.float/float/g' {} +

/content/TrackEval


In [ ]:
%cd /content/TrackEval

!grep -R "np.int" -n trackeval
!find trackeval -type f -name "*.py" -exec sed -i 's/np.int/int/g' {} +

/content/TrackEval


In [ ]:
!python scripts/run_mot_challenge.py \
  --BENCHMARK UA-DETRAC \
  --SPLIT_TO_EVAL train \
  --TRACKERS_TO_EVAL yolo30k_deepsort \
  --METRICS HOTA CLEAR Identity \
  --USE_PARALLEL False


Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
PRINT_CONFIG         : True                          
GT_FOLDER            : /content/TrackEval/data/gt/mot_challenge/
TRACKERS_FOLDER      : /content/TrackEval/data/trackers/mot_challenge/
OUTPUT_FOL